In [2]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, BatchNormalization, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Global Configuration
OUTPUT_DIR = '/kaggle/working/'
print(f"🚀 Job Started. Outputs will be saved to: {OUTPUT_DIR}")

# Helper Function to Find Files Recursively
def find_file(filename_part, search_path='/kaggle/input'):
    for root, dirs, files in os.walk(search_path):
        for file in files:
            if filename_part in file:
                return os.path.join(root, file)
    return None

# ======================================================
# MODULE 1: DIABETES (Stacking Ensemble)
# ======================================================
print("\n" + "="*40)
print("🔹 STARTING MODULE 1: DIABETES MELLITUS")
print("="*40)

# Auto-find the diabetes dataset
diabetes_path = find_file('diabetes.csv')

if diabetes_path:
    print(f"Found Diabetes Dataset: {diabetes_path}")
    df = pd.read_csv(diabetes_path)
    
    # Preprocessing
    zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    df[zero_cols] = df[zero_cols].replace(0, np.nan)
    
    imputer = SimpleImputer(strategy='median')
    df[zero_cols] = imputer.fit_transform(df[zero_cols])
    
    X = df.drop('Outcome', axis=1)
    y = df['Outcome']
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    
    # Stacking Ensemble
    estimators = [
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)),
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)),
        ('svm', SVC(probability=True, kernel='rbf', C=10, gamma='scale', random_state=42))
    ]
    
    clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=5)
    
    print("Training Diabetes Model...")
    clf.fit(X_train, y_train)
    
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"✅ Diabetes Accuracy: {acc*100:.2f}%")
    
    joblib.dump(clf, os.path.join(OUTPUT_DIR, 'diabetes_stacking_model.pkl'))
    joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'diabetes_scaler.pkl'))
else:
    print("❌ Error: 'diabetes.csv' not found. Please check Input files.")

# ======================================================
# MODULE 2: HEART DISEASE (Raw Data Loader)
# ======================================================
print("\n" + "="*40)
print("🔹 STARTING MODULE 2: HEART DISEASE")
print("="*40)

# Custom Loader for Raw MIT-BIH Files (e.g., 100_ekg.csv, 100_annotations.csv)
def load_mit_bih_raw(base_path):
    signals = []
    labels = []
    # Standard MIT-BIH classes mapping
    # N=Normal, S=Supraventricular, V=Ventricular, F=Fusion, Q=Unknown
    classes = {'N': 0, 'S': 1, 'V': 2, 'F': 3, 'Q': 4}
    
    # Find all annotation files first
    annotation_files = glob.glob(os.path.join(base_path, '**', '*_annotations.csv'), recursive=True)
    
    if not annotation_files:
        # Try looking for older format (100.csv) if modern not found
        annotation_files = glob.glob(os.path.join(base_path, '**', '*_annotations_1.csv'), recursive=True)

    print(f"Found {len(annotation_files)} patient records. Processing...")
    
    count = 0
    for ann_file in annotation_files:
        if count >= 10: break # LIMIT TO 10 PATIENTS FOR SPEED IN DEMO (Remove this line for full training)
        
        try:
            # Infer the signal file path from the annotation file name
            # e.g., '100_annotations.csv' -> '100_ekg.csv'
            prefix = ann_file.split('_annotations')[0]
            ekg_file = prefix + '_ekg.csv'
            
            if not os.path.exists(ekg_file):
                # Try finding it recursively if path logic fails
                filename = os.path.basename(prefix) + '_ekg.csv'
                ekg_file = find_file(filename, base_path)
            
            if ekg_file and os.path.exists(ekg_file):
                # Load Data
                ann_df = pd.read_csv(ann_file) # Contains 'index' and 'type'
                ekg_df = pd.read_csv(ekg_file) # Contains signals
                
                # Extract simple heartbeat segments (basic windowing)
                signal_data = ekg_df.iloc[:, 1].values # Usually Lead II is 2nd column
                
                for _, row in ann_df.iterrows():
                    idx = int(row[0]) if isinstance(row[0], (int, float)) else int(row['index'])
                    label_char = row[1] if isinstance(row[1], str) else row['type']
                    
                    # Map label to class
                    if label_char in classes:
                        # Extract 180 samples window (centered)
                        start, end = idx - 90, idx + 90
                        if start >= 0 and end < len(signal_data):
                            beat = signal_data[start:end]
                            signals.append(beat)
                            labels.append(classes[label_char])
                count += 1
        except Exception as e:
            print(f"Skipping file {ann_file}: {e}")
            
    return np.array(signals), np.array(labels)

mit_path = '/kaggle/input/mit-bih-arrhythmia-database-modern-2023'
X_heart, y_heart = load_mit_bih_raw(mit_path)

if len(X_heart) > 0:
    print(f"Processed {len(X_heart)} heartbeats.")
    
    # Reshape for CNN
    X_heart = X_heart.reshape(X_heart.shape[0], X_heart.shape[1], 1)
    y_heart = to_categorical(y_heart, num_classes=5)
    
    X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_heart, y_heart, test_size=0.2, random_state=42)
    
    # Model
    inp = Input(shape=(X_heart.shape[1], 1))
    x = Conv1D(32, 5, activation='relu', padding='same')(inp)
    x = MaxPooling1D(2)(x)
    x = Bidirectional(LSTM(32))(x)
    x = Dense(32, activation='relu')(x)
    out = Dense(5, activation='softmax')(x)
    
    model_h = Model(inputs=inp, outputs=out)
    model_h.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    print("Training Heart Model...")
    model_h.fit(X_train_h, y_train_h, epochs=5, batch_size=32, validation_data=(X_test_h, y_test_h))
    model_h.save(os.path.join(OUTPUT_DIR, 'heart_model.keras'))
    print("✅ Heart Model Saved.")
else:
    print("❌ Error: Could not extract heartbeats. Check if '_ekg.csv' files exist.")

# ======================================================
# MODULE 3: PARKINSON'S (Existing Working Code)
# ======================================================
print("\n" + "="*40)
print("🔹 STARTING MODULE 3: PARKINSON'S DISEASE")
print("="*40)

# Drawings
drawing_path = '/kaggle/input/parkinsons-drawings'
X_img, y_img = [], []
IMG_SIZE = 128
img_files = glob.glob(drawing_path + '/**/*.png', recursive=True) + glob.glob(drawing_path + '/**/*.jpg', recursive=True)

for f in img_files:
    label = 1 if 'parkinson' in f.lower() else 0 if 'healthy' in f.lower() else None
    if label is not None:
        try:
            img = cv2.imread(f)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X_img.append(img)
            y_img.append(label)
        except: pass

if len(X_img) > 0:
    X_img = np.array(X_img) / 255.0
    y_img = np.array(y_img)
    model_img = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        MaxPooling2D(2,2), Flatten(), Dense(64, activation='relu'), Dense(1, activation='sigmoid')
    ])
    model_img.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_img.fit(X_img, y_img, epochs=5, batch_size=32, validation_split=0.2)
    model_img.save(os.path.join(OUTPUT_DIR, 'parkinsons_drawing_model.keras'))
    print("✅ Parkinson's Drawing Model Saved.")

# Voice
voice_file = find_file('parkinsons.data') or find_file('parkinsons.csv')
if voice_file:
    df_voice = pd.read_csv(voice_file)
    if 'name' in df_voice.columns: df_voice = df_voice.drop('name', axis=1)
    X_v = df_voice.drop('status', axis=1)
    y_v = df_voice['status']
    scaler_v = StandardScaler()
    X_v = scaler_v.fit_transform(X_v)
    
    model_voice = Sequential([Dense(64, activation='relu', input_shape=(X_v.shape[1],)), Dense(1, activation='sigmoid')])
    model_voice.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_voice.fit(X_v, y_v, epochs=50, batch_size=16, verbose=0)
    model_voice.save(os.path.join(OUTPUT_DIR, 'parkinsons_voice_model.keras'))
    joblib.dump(scaler_v, os.path.join(OUTPUT_DIR, 'parkinsons_voice_scaler.pkl'))
    print("✅ Parkinson's Voice Model Saved.")

print("\n🎉 DONE! Files are in /kaggle/working/")

🚀 Job Started. Outputs will be saved to: /kaggle/working/

🔹 STARTING MODULE 1: DIABETES MELLITUS
Found Diabetes Dataset: /kaggle/input/d/uciml/pima-indians-diabetes-database/diabetes.csv
Training Diabetes Model...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:35:38] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:35:40] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Diabetes Accuracy: 74.03%

🔹 STARTING MODULE 2: HEART DISEASE
Found 48 patient records. Processing...


/tmp/ipykernel_55/791749769.py:130: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  idx = int(row[0]) if isinstance(row[0], (int, float)) else int(row['index'])
/tmp/ipykernel_55/791749769.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  label_char = row[1] if isinstance(row[1], str) else row['type']
/tmp/ipykernel_55/791749769.py:130: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  idx = int(row[0]) if isinstance(row[0], (int, flo

Processed 16033 heartbeats.
Training Heart Model...
Epoch 1/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9211 - loss: 0.4521 - val_accuracy: 0.9654 - val_loss: 0.1432
Epoch 2/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9737 - loss: 0.1103 - val_accuracy: 0.9769 - val_loss: 0.0999
Epoch 3/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9808 - loss: 0.0822 - val_accuracy: 0.9757 - val_loss: 0.0943
Epoch 4/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9852 - loss: 0.0572 - val_accuracy: 0.9804 - val_loss: 0.0851
Epoch 5/5
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9849 - loss: 0.0604 - val_accuracy: 0.9800 - val_loss: 0.0841
✅ Heart Model Saved.

🔹 STARTING MODULE 3: PARKINSON'S DISEASE


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 200ms/step - accuracy: 1.0000 - loss: 0.1519 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
✅ Parkinson's Drawing Model Saved.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


✅ Parkinson's Voice Model Saved.

🎉 DONE! Files are in /kaggle/working/


In [3]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Configuration
OUTPUT_DIR = '/kaggle/working/'
IMG_SIZE = 224 # VGG16 expects 224x224
print(f"🚀 Job Started. Outputs will be saved to: {OUTPUT_DIR}")

# ======================================================
# SMART DATA LOADER (Fixes the path issue)
# ======================================================
print("\n" + "="*40)
print("🔹 STARTING ROBUST PARKINSON'S MODULE (VGG16)")
print("="*40)

search_path = '/kaggle/input/parkinsons-drawings'
print(f"Scanning {search_path} for images...")

# Recursively find all PNG/JPG images
all_images = glob.glob(search_path + '/**/*.png', recursive=True) + \
             glob.glob(search_path + '/**/*.jpg', recursive=True) + \
             glob.glob(search_path + '/**/*.jpeg', recursive=True)

print(f"Found {len(all_images)} total images.")

X = []
y = []

# Logic: Look for keywords in the FILE PATH to determine label
# This works regardless of whether the folder is named "wave", "waves", "spiral", etc.
count_healthy = 0
count_parkinson = 0

for img_path in all_images:
    label = None
    path_lower = img_path.lower()
    
    # Robust Labeling Logic
    if 'parkinson' in path_lower:
        label = 1
    elif 'healthy' in path_lower or 'control' in path_lower:
        label = 0
        
    if label is not None:
        try:
            # Load and Resize for VGG16
            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X.append(img)
            y.append(label)
            
            if label == 1: count_parkinson += 1
            else: count_healthy += 1
        except Exception as e:
            print(f"Skipping broken image: {img_path}")

print(f"✅ Data Loaded: {count_parkinson} Parkinson's vs {count_healthy} Healthy images.")

if len(X) == 0:
    print("❌ ERROR: No images found! Check if the dataset is added to the notebook.")
else:
    X = np.array(X)
    y = np.array(y)

    # Split Data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # ======================================================
    # DATA AUGMENTATION (Prevents Overfitting)
    # ======================================================
    # This creates "fake" variations (zooms, rotations) to make the model harder to fool
    datagen = ImageDataGenerator(
        rotation_range=20,      # Rotate slightly (tremors look different at angles)
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,         # Zoom in/out
        horizontal_flip=False,  # Don't flip (handwriting direction matters!)
        vertical_flip=False
    )
    
    datagen.fit(X_train)

    # ======================================================
    # TRANSFER LEARNING MODEL (VGG16)
    # ======================================================
    print("Loading VGG16 (Pre-trained on ImageNet)...")
    
    # 1. Load VGG16 without the top classification layers
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    
    # 2. Freeze the base layers (we don't want to ruin the pre-trained weights)
    for layer in base_model.layers:
        layer.trainable = False
        
    # 3. Add our Custom Layers for Parkinson's
    x = base_model.output
    x = GlobalAveragePooling2D()(x) # Better than Flatten for VGG
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x) # High dropout to fight 100% overfitting
    predictions = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    
    # 4. Compile
    # Use a small learning rate (0.0001) because VGG is sensitive
    model.compile(optimizer=Adam(learning_rate=0.0001), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])

    # 5. Train with Callbacks
    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        ModelCheckpoint(os.path.join(OUTPUT_DIR, 'parkinsons_vgg16_best.keras'), save_best_only=True)
    ]

    print("Training VGG16...")
    history = model.fit(
        datagen.flow(X_train, y_train, batch_size=32),
        validation_data=(X_test, y_test),
        epochs=30, # Allow more epochs, EarlyStopping will cut it short if needed
        callbacks=callbacks
    )
    
    # 6. Final Evaluation
    print("\nEvaluating on Test Set...")
    loss, acc = model.evaluate(X_test, y_test)
    print(f"🏆 Final Robust Accuracy: {acc*100:.2f}%")
    
    if acc > 0.99:
        print("⚠️ Note: Accuracy is still very high. This is likely due to the small size of the public dataset.")
        print("   However, using VGG16 + Augmentation makes this model much more scientifically valid than the previous one.")

🚀 Job Started. Outputs will be saved to: /kaggle/working/

🔹 STARTING ROBUST PARKINSON'S MODULE (VGG16)
Scanning /kaggle/input/parkinsons-drawings for images...
Found 408 total images.
✅ Data Loaded: 408 Parkinson's vs 0 Healthy images.
Loading VGG16 (Pre-trained on ImageNet)...
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training VGG16...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step - accuracy: 0.4578 - loss: 2.6672 - val_accuracy: 0.7195 - val_loss: 0.4852
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 372ms/step - accuracy: 0.8607 - loss: 0.4725 - val_accuracy: 0.9878 - val_loss: 0.0366
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 422ms/step - accuracy: 0.8750 - loss: 0.3657 - val_accuracy: 1.0000 - val_loss: 0.0044
Epoch 4/30
 3/11 ━━━━━━━━━━━━━━━━━━━━ 2s 305ms/step - accuracy: 0.8663 - loss: 0.4009

KeyboardInterrupt: 

In [1]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input, GlobalAveragePooling2D, Conv1D, MaxPooling1D, Bidirectional, LSTM, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Global Config
OUTPUT_DIR = '/kaggle/working/'
IMG_SIZE = 224 # Required for VGG16
print(f"🚀 Job Started. Outputs will be saved to: {OUTPUT_DIR}")

# ======================================================
# MODULE 1: PARKINSON'S DISEASE (VGG16 Transfer Learning)
# ======================================================
print("\n" + "="*40)
print("🔹 MODULE 1: PARKINSON'S (VGG16 Professional Model)")
print("="*40)

# 1. robust Image Loader (Handling 'spiral', 'wave', 'training', 'testing' folders)
search_path = '/kaggle/input/parkinsons-drawings'
image_files = glob.glob(search_path + '/**/*.png', recursive=True) + \
              glob.glob(search_path + '/**/*.jpg', recursive=True)

X_img, y_img = [], []
print(f"Found {len(image_files)} images. Loading...")

for f in image_files:
    # Labeling Logic: Check if file path contains specific keywords
    # This works for: .../spiral/training/parkinson/img.png
    label = None
    if 'parkinson' in f.lower():
        label = 1
    elif 'healthy' in f.lower():
        label = 0
    
    if label is not None:
        try:
            img = cv2.imread(f)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) # Resize to 224x224
            X_img.append(img)
            y_img.append(label)
        except: pass

if len(X_img) > 0:
    X_img = np.array(X_img)
    y_img = np.array(y_img)
    print(f"Loaded {len(X_img)} valid images.")

    # 2. Split Data (Stratified to keep balance)
    X_train, X_test, y_train, y_test = train_test_split(X_img, y_img, test_size=0.2, random_state=42, stratify=y_img)

    # 3. Data Augmentation (Crucial to prevent fake 100% accuracy)
    datagen = ImageDataGenerator(
        rotation_range=15, width_shift_range=0.1, height_shift_range=0.1, 
        zoom_range=0.1, horizontal_flip=False, vertical_flip=False
    )
    datagen.fit(X_train)

    # 4. Build VGG16 Model
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    
    # Freeze base layers (use pre-trained knowledge)
    for layer in base_model.layers:
        layer.trainable = False
        
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x) # Strong dropout to prevent overfitting
    predictions = Dense(1, activation='sigmoid')(x)
    
    model_pd = Model(inputs=base_model.input, outputs=predictions)
    
    # Low learning rate for transfer learning
    model_pd.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

    # 5. Train
    print("Training VGG16 on Drawings...")
    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        ModelCheckpoint(os.path.join(OUTPUT_DIR, 'parkinsons_vgg16.keras'), save_best_only=True, monitor='val_accuracy')
    ]
    
    model_pd.fit(datagen.flow(X_train, y_train, batch_size=32), 
                 validation_data=(X_test, y_test), epochs=20, callbacks=callbacks)
    
    loss, acc = model_pd.evaluate(X_test, y_test)
    print(f"🏆 Parkinson's Image Accuracy: {acc*100:.2f}%")
else:
    print("❌ Error: No images loaded.")

# --- Parkinson's Voice (Tabular) ---
print("Training Voice Model...")
voice_files = glob.glob('/kaggle/input/**/parkinsons.data', recursive=True)
if voice_files:
    df_v = pd.read_csv(voice_files[0])
    if 'name' in df_v.columns: df_v = df_v.drop('name', axis=1)
    X_v = df_v.drop('status', axis=1)
    y_v = df_v['status']
    scaler_v = StandardScaler()
    X_v = scaler_v.fit_transform(X_v)
    
    model_voice = Sequential([Dense(64, activation='relu', input_shape=(X_v.shape[1],)), Dense(1, activation='sigmoid')])
    model_voice.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_voice.fit(X_v, y_v, epochs=50, batch_size=16, verbose=0)
    model_voice.save(os.path.join(OUTPUT_DIR, 'parkinsons_voice_model.keras'))
    joblib.dump(scaler_v, os.path.join(OUTPUT_DIR, 'parkinsons_voice_scaler.pkl'))
    print("✅ Voice Model Saved.")

# ======================================================
# MODULE 2: DIABETES (Stacking Ensemble)
# ======================================================
print("\n" + "="*40)
print("🔹 MODULE 2: DIABETES (Robust Search)")
print("="*40)

# Recursive finder for diabetes.csv
diabetes_files = glob.glob('/kaggle/input/**/diabetes.csv', recursive=True)

if diabetes_files:
    print(f"Found: {diabetes_files[0]}")
    df = pd.read_csv(diabetes_files[0])
    
    # Clean Data
    zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    df[zero_cols] = df[zero_cols].replace(0, np.nan)
    imputer = SimpleImputer(strategy='median')
    df[zero_cols] = imputer.fit_transform(df[zero_cols])
    
    X = df.drop('Outcome', axis=1)
    y = df['Outcome']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    
    # Stacking
    estimators = [
        ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)),
        ('svm', SVC(probability=True, kernel='rbf', random_state=42))
    ]
    clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=5)
    
    print("Training Diabetes Stacking Model...")
    clf.fit(X_train, y_train)
    print(f"🏆 Diabetes Accuracy: {accuracy_score(y_test, clf.predict(X_test))*100:.2f}%")
    
    joblib.dump(clf, os.path.join(OUTPUT_DIR, 'diabetes_stacking_model.pkl'))
    joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'diabetes_scaler.pkl'))
else:
    print("❌ Error: diabetes.csv not found.")

# ======================================================
# MODULE 3: HEART DISEASE (Raw MIT-BIH Loader)
# ======================================================
print("\n" + "="*40)
print("🔹 MODULE 3: HEART DISEASE (Raw Data Processing)")
print("="*40)

mit_base = '/kaggle/input/mit-bih-arrhythmia-database-modern-2023'
signals, labels = [], []
classes = {'N': 0, 'S': 1, 'V': 2, 'F': 3, 'Q': 4}

# Find all annotation files
ann_files = glob.glob(mit_base + '/**/*_annotations.csv', recursive=True) + \
            glob.glob(mit_base + '/**/*_annotations_1.csv', recursive=True)

print(f"Found {len(ann_files)} patient records. Processing max 15 for demo speed...")

count = 0
for ann_file in ann_files:
    if count >= 15: break # Limit for speed, remove for full training
    try:
        # Infer EKG file
        prefix = ann_file.split('_annotations')[0]
        ekg_file = prefix + '_ekg.csv'
        
        if os.path.exists(ekg_file):
            ann_df = pd.read_csv(ann_file)
            ekg_df = pd.read_csv(ekg_file)
            sig_data = ekg_df.iloc[:, 1].values # Lead II
            
            for _, row in ann_df.iterrows():
                idx = int(row[0]) if isinstance(row[0], (int, float)) else int(row['index'])
                typ = row[1] if isinstance(row[1], str) else row['type']
                
                if typ in classes:
                    start, end = idx - 90, idx + 90
                    if start >= 0 and end < len(sig_data):
                        signals.append(sig_data[start:end])
                        labels.append(classes[typ])
            count += 1
    except: pass

if len(signals) > 0:
    X_heart = np.array(signals).reshape(len(signals), 180, 1)
    y_heart = to_categorical(labels, num_classes=5)
    X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_heart, y_heart, test_size=0.2, random_state=42)
    
    # CNN-BiLSTM
    inp = Input(shape=(180, 1))
    x = Conv1D(64, 5, activation='relu', padding='same')(inp)
    x = MaxPooling1D(2)(x)
    x = Bidirectional(LSTM(64))(x)
    x = Dense(64, activation='relu')(x)
    out = Dense(5, activation='softmax')(x)
    
    model_h = Model(inputs=inp, outputs=out)
    model_h.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    print("Training Heart Model...")
    model_h.fit(X_train_h, y_train_h, epochs=10, batch_size=64, validation_data=(X_test_h, y_test_h))
    model_h.save(os.path.join(OUTPUT_DIR, 'heart_model.keras'))
    print("✅ Heart Model Saved.")
else:
    print("❌ Error: No heartbeats extracted.")

print("\n🎉 ALL DONE! Check Output folder.")

2026-01-29 16:42:35.060410: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769704955.255012      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769704955.306891      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769704955.727676      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769704955.727716      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769704955.727719      55 computation_placer.cc:177] computation placer alr

🚀 Job Started. Outputs will be saved to: /kaggle/working/

🔹 MODULE 1: PARKINSON'S (VGG16 Professional Model)
Found 408 images. Loading...
Loaded 408 valid images.


I0000 00:00:1769704973.852831      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1769704973.859002      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training VGG16 on Drawings...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


...

✅ Voice Model Saved.

🔹 MODULE 2: DIABETES (Robust Search)
Found: /kaggle/input/d/uciml/pima-indians-diabetes-database/diabetes.csv
Training Diabetes Stacking Model...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:44:45] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:44:46] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:44:47] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🏆 Diabetes Accuracy: 75.32%

🔹 MODULE 3: HEART DISEASE (Raw Data Processing)
Found 48 patient records. Processing max 15 for demo speed...


/tmp/ipykernel_55/2602549586.py:201: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  idx = int(row[0]) if isinstance(row[0], (int, float)) else int(row['index'])
/tmp/ipykernel_55/2602549586.py:202: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  typ = row[1] if isinstance(row[1], str) else row['type']
/tmp/ipykernel_55/2602549586.py:201: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  idx = int(row[0]) if isinstance(row[0], (int, float))

Training Heart Model...
Epoch 1/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9147 - loss: 0.4308 - val_accuracy: 0.9686 - val_loss: 0.1038
Epoch 2/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9730 - loss: 0.0878 - val_accuracy: 0.9827 - val_loss: 0.0749
Epoch 3/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9794 - loss: 0.0700 - val_accuracy: 0.9847 - val_loss: 0.0591
Epoch 4/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9833 - loss: 0.0576 - val_accuracy: 0.9843 - val_loss: 0.0608
Epoch 5/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9847 - loss: 0.0543 - val_accuracy: 0.9861 - val_loss: 0.0619
Epoch 6/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9844 - loss: 0.0537 - val_accuracy: 0.9855 - val_loss: 0.0592
Epoch 7/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9858 - loss: 0.0488 - val_accuracy: 0.9861 - val_loss: 0.0546
Epoch 8/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9882 -

In [4]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Suppress Warnings (Clean Output)
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Machine Learning Imports
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from xgboost import XGBClassifier

# Deep Learning Imports (Keras/TensorFlow)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, Conv2D, MaxPooling2D, GaussianNoise, Input, 
    Conv1D, MaxPooling1D, Bidirectional, LSTM, BatchNormalization, 
    GlobalAveragePooling2D
)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import AdamW  # <--- UPGRADED OPTIMIZER
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Global Config
OUTPUT_DIR = '/kaggle/working/'
IMG_SIZE = 128
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

print(f"🚀 Job Started. Using TensorFlow {tf.__version__} with AdamW Optimizer.")

# Helper to save plots
def save_cm_plot(y_test, y_pred, title, filename, labels):
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.close()

def save_training_curve(history, title, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--')
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.close()

# ======================================================
# 1. DIABETES MODULE (Real Training with SMOTE)
# ======================================================
print("\n" + "="*50)
print("🔹 MODULE 1: DIABETES (Stacking + SMOTE)")
print("="*50)

diabetes_files = glob.glob('/kaggle/input/**/diabetes.csv', recursive=True)

if diabetes_files:
    # A. Load & Preprocess
    df = pd.read_csv(diabetes_files[0])
    zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    df[zero_cols] = df[zero_cols].replace(0, np.nan)
    df[zero_cols] = SimpleImputer(strategy='median').fit_transform(df[zero_cols])
    
    X = df.drop('Outcome', axis=1)
    y = df['Outcome']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # B. SMOTE Balancing (Fixes Imbalance)
    print("Applying SMOTE balancing...")
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_scaled, y)
    
    # C. Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)
    
    # D. Train Stacking Ensemble
    estimators = [
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('xgb', XGBClassifier(eval_metric='logloss', use_label_encoder=False)),
        ('svm', SVC(probability=True, kernel='rbf'))
    ]
    clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=5)
    print("Training Stacking Ensemble...")
    clf.fit(X_train, y_train)
    
    # E. Real Evaluation
    y_pred = clf.predict(X_test)
    dia_acc = accuracy_score(y_test, y_pred)
    print(f"✅ Diabetes Real Accuracy: {dia_acc*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=['Healthy', 'Diabetic']))
    
    # F. Generate Graphs
    save_cm_plot(y_test, y_pred, f"Diabetes Confusion Matrix\n(Stacking Ensemble + SMOTE)", "diabetes_cm.png", ['Healthy', 'Diabetic'])
    
    joblib.dump(clf, os.path.join(OUTPUT_DIR, 'diabetes_stacking_model.pkl'))
    joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'diabetes_scaler.pkl'))

# ======================================================
# 2. HEART DISEASE MODULE (Real Training with Bi-LSTM)
# ======================================================
print("\n" + "="*50)
print("🔹 MODULE 2: HEART DISEASE (CNN-BiLSTM)")
print("="*50)

mit_base = '/kaggle/input/mit-bih-arrhythmia-database-modern-2023'
signals, labels = [], []
classes = {'N': 0, 'S': 1, 'V': 2, 'F': 3, 'Q': 4}
ann_files = glob.glob(mit_base + '/**/*_annotations.csv', recursive=True) + glob.glob(mit_base + '/**/*_annotations_1.csv', recursive=True)

# Process Data
print(f"Found {len(ann_files)} patient files. Processing subset for demo...")
count = 0
for ann_file in ann_files:
    if count >= 25: break  # Limit to 25 patients for reasonable run-time
    try:
        prefix = ann_file.split('_annotations')[0]
        ekg_file = prefix + '_ekg.csv'
        if os.path.exists(ekg_file):
            ann_df = pd.read_csv(ann_file)
            ekg_df = pd.read_csv(ekg_file)
            sig_data = ekg_df.iloc[:, 1].values
            
            # Robust Iteration
            for i in range(len(ann_df)):
                row = ann_df.iloc[i]
                idx = int(row.iloc[0]) 
                typ = row.iloc[1]
                
                if typ in classes:
                    start, end = idx - 90, idx + 90
                    if start >= 0 and end < len(sig_data):
                        signals.append(sig_data[start:end])
                        labels.append(classes[typ])
            count += 1
    except: pass

if len(signals) > 0:
    X_heart = np.array(signals).reshape(len(signals), 180, 1)
    y_heart = to_categorical(labels, num_classes=5)
    X_train, X_test, y_train, y_test = train_test_split(X_heart, y_heart, test_size=0.2, random_state=42)
    
    # Model Architecture (CNN + Bi-LSTM)
    inp = Input(shape=(180, 1))
    x = Conv1D(32, 5, activation='relu', padding='same')(inp)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)             # <--- Now Imported Correctly
    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    out = Dense(5, activation='softmax')(x)
    
    model_h = Model(inputs=inp, outputs=out)
    
    # Using AdamW for better convergence
    model_h.compile(optimizer=AdamW(learning_rate=0.001, weight_decay=0.004), 
                    loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Real Training
    print("Training Heart Model...")
    history_h = model_h.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test), verbose=1)
    
    # Save Graphs
    save_training_curve(history_h, "Heart Disease Learning Curve\n(Hybrid CNN-BiLSTM)", "heart_curve.png")
    
    y_pred_h_prob = model_h.predict(X_test)
    y_pred_h = np.argmax(y_pred_h_prob, axis=1)
    y_test_h = np.argmax(y_test, axis=1)
    
    save_cm_plot(y_test_h, y_pred_h, "Heart Disease Confusion Matrix", "heart_cm.png", ['N', 'S', 'V', 'F', 'Q'])
    
    model_h.save(os.path.join(OUTPUT_DIR, 'heart_model.keras'))
    print(f"✅ Heart Disease Model Saved. Val Acc: {history_h.history['val_accuracy'][-1]*100:.2f}%")

# ======================================================
# 3. PARKINSON'S MODULE (Real Training VGG16)
# ======================================================
print("\n" + "="*50)
print("🔹 MODULE 3: PARKINSON'S (VGG16 + Augmentation)")
print("="*50)

search_path = '/kaggle/input/parkinsons-drawings'
img_files = glob.glob(search_path + '/**/*.png', recursive=True) + glob.glob(search_path + '/**/*.jpg', recursive=True)
X_img, y_img = [], []

print(f"Found {len(img_files)} images...")
for f in img_files:
    label = 1 if 'parkinson' in f.lower() else 0 if 'healthy' in f.lower() else None
    if label is not None:
        try:
            img = cv2.imread(f)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X_img.append(img)
            y_img.append(label)
        except: pass

if len(X_img) > 0:
    X_img = np.array(X_img) / 255.0
    y_img = np.array(y_img)
    X_train, X_test, y_train, y_test = train_test_split(X_img, y_img, test_size=0.2, random_state=42, stratify=y_img)
    
    # Augmentation
    datagen = ImageDataGenerator(
        rotation_range=20, 
        zoom_range=0.2, 
        width_shift_range=0.1, 
        height_shift_range=0.1
    )
    datagen.fit(X_train)
    
    # Model (Transfer Learning)
    base = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    for l in base.layers: l.trainable = False
    
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation='sigmoid')(x)
    
    model_p = Model(inputs=base.input, outputs=out)
    
    # Using AdamW
    model_p.compile(optimizer=AdamW(learning_rate=0.0001, weight_decay=1e-5), 
                    loss='binary_crossentropy', metrics=['accuracy'])
    
    # Real Training
    print("Training Parkinson's Model...")
    history_p = model_p.fit(datagen.flow(X_train, y_train, batch_size=32), 
                            validation_data=(X_test, y_test), 
                            epochs=15, verbose=1)
    
    # Save Graphs
    save_training_curve(history_p, "Parkinson's Learning Curve\n(VGG16 Transfer Learning)", "parkinsons_curve.png")
    
    y_pred_p = (model_p.predict(X_test) > 0.5).astype(int)
    save_cm_plot(y_test, y_pred_p, "Parkinson's Confusion Matrix\n(Augmented Data)", "parkinsons_cm.png", ['Healthy', 'Parkinson'])
    
    model_p.save(os.path.join(OUTPUT_DIR, 'parkinsons_vgg16.keras'))
    print(f"✅ Parkinson's Model Saved. Val Acc: {history_p.history['val_accuracy'][-1]*100:.2f}%")

print("\n🎉 ALL TASKS COMPLETED SUCCESSFULLY. CHECK OUTPUT FOLDER.")

🚀 Job Started. Using TensorFlow 2.19.0 with AdamW Optimizer.

🔹 MODULE 1: DIABETES (Stacking + SMOTE)
Applying SMOTE balancing...
Training Stacking Ensemble...
✅ Diabetes Real Accuracy: 80.50%
              precision    recall  f1-score   support

     Healthy       0.83      0.76      0.79        99
    Diabetic       0.78      0.85      0.82       101

    accuracy                           0.81       200
   macro avg       0.81      0.80      0.80       200
weighted avg       0.81      0.81      0.80       200


🔹 MODULE 2: HEART DISEASE (CNN-BiLSTM)
Found 48 patient files. Processing subset for demo...
Training Heart Model...
Epoch 1/10


I0000 00:00:1769709318.080284     134 cuda_dnn.cc:529] Loaded cuDNN version 91002


509/509 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.9302 - loss: 0.2744 - val_accuracy: 0.9791 - val_loss: 0.0710
Epoch 2/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9775 - loss: 0.0817 - val_accuracy: 0.9856 - val_loss: 0.0485
Epoch 3/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9842 - loss: 0.0614 - val_accuracy: 0.9881 - val_loss: 0.0448
Epoch 4/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9855 - loss: 0.0514 - val_accuracy: 0.9881 - val_loss: 0.0480
Epoch 5/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9859 - loss: 0.0537 - val_accuracy: 0.9897 - val_loss: 0.0381
Epoch 6/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9883 - loss: 0.0424 - val_accuracy: 0.9888 - val_loss: 0.0414
Epoch 7/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9885 - loss: 0.0419 - val_accuracy: 0.9907 - val_loss: 0.0352
Epoch 8/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9886 - loss: 0.0397 - val_accuracy: 0.99

I0000 00:00:1769709398.263997     135 service.cc:152] XLA service 0x7e62c80e91b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769709398.264049     135 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1769709398.264055     135 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5


 1/11 ━━━━━━━━━━━━━━━━━━━━ 51s 5s/step - accuracy: 1.0000 - loss: 0.2443

I0000 00:00:1769709402.141460     135 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


11/11 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.9429 - loss: 0.3207 - val_accuracy: 1.0000 - val_loss: 0.1998
Epoch 2/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.9794 - loss: 0.2348 - val_accuracy: 1.0000 - val_loss: 0.1214
Epoch 3/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 134ms/step - accuracy: 1.0000 - loss: 0.1382 - val_accuracy: 1.0000 - val_loss: 0.0785
Epoch 4/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 132ms/step - accuracy: 1.0000 - loss: 0.0844 - val_accuracy: 1.0000 - val_loss: 0.0544
Epoch 5/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step - accuracy: 1.0000 - loss: 0.0648 - val_accuracy: 1.0000 - val_loss: 0.0400
Epoch 6/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step - accuracy: 1.0000 - loss: 0.0490 - val_accuracy: 1.0000 - val_loss: 0.0306
Epoch 7/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 139ms/step - accuracy: 1.0000 - loss: 0.0339 - val_accuracy: 1.0000 - val_loss: 0.0245
Epoch 8/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 1.0000 - loss: 0.0300 - val_accuracy: 1.0000 - val_lo